### # **Task 1: Write Staging — Build finance_fin_metrics — the full-refresh staging query**

In [0]:
%sql
    
-- ============================================================================
-- TASK 1: WRITE STAGING
-- ============================================================================
-- PURPOSE  : Full refresh of finance_fin_metrics for the Finance team.
--            Reads acquisitions, SKU hierarchy, delivered revenue (RR), and
--            VAS data, computes financial metrics, acquisition types, and
--            asset costs, then writes the complete result to a staging table.
--
-- WHY STAGING : We never write directly to the live table. If this query
--               fails mid-run, the live table stays untouched. Task 2 validates
--               the staging output before Task 3 promotes it to live.
--
-- SCHEDULE    : Runs every 4 hours via Databricks Workflow (0 */4 * * *)
-- DEPENDS ON  : Nothing — this is the entry point task
-- NEXT TASK   : task2_validate_staging.sql
-- ===========================================================================

CREATE OR REPLACE TABLE furlenco_analytics.materialized_tables.finance_fin_metrics_staging AS

WITH RECURSIVE sku_hierarchy (catalog_product_id, current_sku_id, sku_type, quantity) AS (
    -- Anchor Member
    SELECT
        pr.id,
        skus.id,
        skus.sku_type,
        1 AS quantity
    FROM furlenco_silver.plutus_evolve.Products pr
    JOIN furlenco_silver.wmsl_evolve.skus skus ON skus.catalog_product_id = pr.id

    UNION ALL

    -- Recursive Member
    SELECT
        sh.catalog_product_id,
        comp.sku_id,
        child_skus.sku_type,
        comp.quantity
    FROM sku_hierarchy sh
    JOIN furlenco_silver.wmsl_evolve.Sku_Composition_Units comp ON comp.parent_sku_id = sh.current_sku_id
    JOIN furlenco_silver.wmsl_evolve.skus child_skus ON child_skus.id = comp.sku_id
    WHERE sh.sku_type IN ('ITEM', 'ATTACHMENT')
)

, product_base_price AS (
    SELECT
        sh.catalog_product_id,
        SUM(COALESCE(abp.unit_price * sh.quantity, 0)) AS unit_price
    FROM sku_hierarchy sh
    LEFT JOIN furlenco_silver.wms_evolve.asset asset ON asset.wmsl_sku_id = sh.current_sku_id
    LEFT JOIN furlenco_silver.wms_evolve.asset_base_prices abp ON abp.asset_id = asset.id
    WHERE sh.sku_type = 'WAREHOUSE_SKU'
    GROUP BY 1
)

, base_return_item AS (
    SELECT
        items.id                                                                       AS item_id,
        items.name                                                                     AS product_name,
        items.order_id,
        catalog_item_id,
        CAST(items.user_details_displayid AS STRING)                                   AS fur_id,
        items.user_id,
        items.Display_id                                                               AS item_display_id,
        ord.Display_id                                                                 AS ord_display_id,
        items.activation_date,
        CAST(items.logistics_attributes_snapshot_totalvolumeincft AS STRING)           AS cft,
        from_utc_timestamp(items.created_at, 'Asia/Kolkata')                           AS item_created_at,
        MIN(CASE WHEN return_items.state <> 'CANCELLED'
                 THEN from_utc_timestamp(return_items.created_at, 'Asia/Kolkata') END) AS return_date,
        MIN(CASE WHEN rtp_orders.state NOT IN ('CANCELLED')
                 THEN from_utc_timestamp(rtp_orders.created_at, 'Asia/Kolkata') END)   AS rent_to_purchase_date,
        items.state                                                                    AS current_entity_state,
        items.delivery_date,
        CASE WHEN items.bundle_id IS NOT NULL                                          THEN items.bundle_id
             WHEN items.bundle_id IS NULL AND items.composite_item_id IS NOT NULL      THEN items.composite_item_id
             ELSE items.id
        END                                                                            AS product_entity_id,
        CASE WHEN items.bundle_id IS NOT NULL                                          THEN 'BUNDLE'
             WHEN items.bundle_id IS NULL AND items.composite_item_id IS NOT NULL      THEN 'COMPOSITE_ITEM'
             ELSE 'ITEM'
        END                                                                            AS product_entity_type
    FROM furlenco_silver.order_management_systems_evolve.items AS items
    LEFT JOIN furlenco_silver.order_management_systems_evolve.orders AS ord
        ON items.order_id = ord.id
    LEFT JOIN furlenco_silver.order_management_systems_evolve.return_items AS return_items
        ON items.id = return_items.item_id
        AND return_items.state <> 'CANCELLED'
    LEFT JOIN furlenco_silver.order_management_systems_evolve.rent_to_purchase_items AS rtp_items
        ON items.id = rtp_items.item_id
    LEFT JOIN furlenco_silver.order_management_systems_evolve.rent_to_purchase_orders AS rtp_orders
        ON rtp_items.rent_to_purchase_order_id = rtp_orders.id
        AND rtp_orders.state NOT IN ('CANCELLED')
    WHERE items.vertical = 'FURLENCO_RENTAL'
      AND items.state NOT IN ('CANCELLED')
      AND ord.state NOT IN ('CANCELLED', 'AWAITING_PAYMENT')
    GROUP BY
        items.id, items.name, items.order_id, catalog_item_id,
        CAST(items.user_details_displayid AS STRING),
        items.user_id, items.activation_date,
        CAST(items.logistics_attributes_snapshot_totalvolumeincft AS STRING),
        from_utc_timestamp(items.created_at, 'Asia/Kolkata'),
        items.Display_id, ord.Display_id,
        items.state, items.delivery_date,
        CASE WHEN items.bundle_id IS NOT NULL                                     THEN items.bundle_id
             WHEN items.bundle_id IS NULL AND items.composite_item_id IS NOT NULL THEN items.composite_item_id
             ELSE items.id END,
        CASE WHEN items.bundle_id IS NOT NULL                                     THEN 'BUNDLE'
             WHEN items.bundle_id IS NULL AND items.composite_item_id IS NOT NULL THEN 'COMPOSITE_ITEM'
             ELSE 'ITEM' END
)

, new_acquisitions AS (
    SELECT
        user_id, item_id, product_name, order_id, catalog_item_id,
        product_entity_id, product_entity_type, current_entity_state, delivery_date,
        fur_id, activation_date, item_created_at,
        COALESCE(return_date, rent_to_purchase_date) AS termination_date,
        DENSE_RANK() OVER (PARTITION BY user_id ORDER BY order_id) AS rnk,
        item_display_id, ord_display_id, cft
    FROM base_return_item
)

-- Pre-aggregate to avoid a correlated subquery (re-scans CTE per row in Spark)
, prior_active_flag AS (
    SELECT DISTINCT a.user_id, a.order_id
    FROM new_acquisitions a
    JOIN new_acquisitions b
        ON  b.user_id  = a.user_id
        AND b.order_id < a.order_id
        AND b.termination_date IS NULL
)

, has_previous_item AS (
    SELECT
        na.*,
        MAX(na.termination_date) OVER (
            PARTITION BY na.user_id
            ORDER BY na.order_id, na.termination_date DESC
            ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING
        ) AS previous_termination_date,
        CASE WHEN paf.user_id IS NOT NULL THEN 1 ELSE 0 END AS has_active_previous_items_corr
    FROM new_acquisitions na
    LEFT JOIN prior_active_flag paf
        ON paf.user_id = na.user_id AND paf.order_id = na.order_id
)

-- Window function alias cannot be referenced in same SELECT level in Databricks;
-- compute max_previous_termination_date in a subquery first.
, acquisition_types AS (
    SELECT
        fur_id, user_id, order_id, catalog_item_id, item_id, product_name,
        activation_date, product_entity_id, product_entity_type, current_entity_state,
        delivery_date, item_created_at, termination_date,
        has_active_previous_items_corr,
        max_previous_termination_date,
        CASE WHEN (
                (item_created_at > max_previous_termination_date
                 AND max_previous_termination_date IS NOT NULL
                 AND has_active_previous_items_corr = 0)
                OR rnk = 1
             ) THEN 'New' ELSE 'Upsell'
        END AS acquisition_type,
        rnk, item_display_id, ord_display_id, cft
    FROM (
        SELECT
            *,
            MIN(previous_termination_date) OVER (PARTITION BY user_id, order_id) AS max_previous_termination_date
        FROM has_previous_item
    )
)

, subscription_groups AS (
    SELECT
        a.*,
        SUM(CASE WHEN a.acquisition_type = 'New' THEN 1 ELSE 0 END) OVER (
            PARTITION BY a.user_id
            ORDER BY a.order_id
            ROWS UNBOUNDED PRECEDING
        ) AS acquisition_group_id
    FROM acquisition_types a
)

, asb_calculation AS (
    SELECT
        s.*,
        FIRST_VALUE(order_id) OVER (
            PARTITION BY user_id, acquisition_group_id
            ORDER BY order_id ASC
        ) AS first_order_in_group
    FROM subscription_groups s
)

, acquisitions AS (
    SELECT
        fur_id, user_id, order_id, catalog_item_id, item_id, product_name,
        activation_date, product_entity_id, product_entity_type, current_entity_state,
        delivery_date, item_created_at, termination_date,
        acquisition_type, acquisition_group_id, first_order_in_group,
        MIN(activation_date) OVER (PARTITION BY fur_id, first_order_in_group) AS acquisition_date,
        item_display_id, ord_display_id, cft
    FROM asb_calculation
)

----------------------------------------------------------------------------------------------
-- Delivered Revenue
----------------------------------------------------------------------------------------------
, transitions AS (
    SELECT
        entity_type, entity_id, st.created_at,
        snapshot_tenureinmonths,
        variant_get(snapshot_paymentdetails_paid_paymentoffers, '$[0].type', 'STRING') AS Payment_offer_flag
    FROM furlenco_silver.order_management_systems_evolve.state_transitions AS st
    JOIN furlenco_silver.order_management_systems_evolve.Events AS et ON st.event_id = et.id
    WHERE entity_type IN ('ITEM', 'ATTACHMENT')
      AND CAST(snapshot_vertical AS STRING) = 'FURLENCO_RENTAL'
      AND st.created_at >= '2025-01-01 00:00:00'::timestamp - interval '330 minutes'
      AND et.name = 'ORDER_PAYMENT_SUCCEEDED'
)

, base_item AS (
    SELECT
        acq.fur_id, acq.user_id, acq.order_id,
        acq.item_id                                                                    AS accountable_entity_id,
        'ITEM'                                                                         AS accountable_entity_type,
        acq.activation_date, acq.first_order_in_group, acq.acquisition_date,
        acq.acquisition_type, acq.catalog_item_id, acq.product_name,
        CAST(snapshot_tenureinmonths AS STRING)                                        AS tenures,
        acq.product_entity_id, acq.product_entity_type, acq.current_entity_state,
        acq.delivery_date,
        Payment_offer_flag,
        acq.cft
    FROM acquisitions AS acq
    LEFT JOIN (
        SELECT * FROM transitions WHERE entity_type = 'ITEM'
    ) AS ist ON ist.entity_id = acq.item_id

    UNION ALL

    SELECT DISTINCT
        CAST(at.user_details_displayid AS STRING)                                      AS fur_id,
        at.user_id, at.order_id,
        at.id                                                                          AS accountable_entity_id,
        'ATTACHMENT'                                                                   AS accountable_entity_type,
        at.activation_date, acq.first_order_in_group, acq.acquisition_date,
        acq.acquisition_type, at.catalog_item_id,
        at.name                                                                        AS product_name,
        CAST(snapshot_tenureinmonths AS STRING)                                        AS tenures,
        CASE WHEN at.composite_item_id IS NOT NULL THEN at.composite_item_id
             ELSE at.id END                                                            AS product_entity_id,
        CASE WHEN at.composite_item_id IS NOT NULL THEN 'COMPOSITE_ITEM'
             ELSE 'ATTACHMENT' END                                                     AS product_entity_type,
        at.state                                                                       AS current_entity_state,
        at.delivery_date,
        Payment_offer_flag,
        CAST(at.logistics_attributes_snapshot_totalvolumeincft AS STRING)              AS cft
    FROM acquisitions AS acq
    LEFT JOIN furlenco_silver.order_management_systems_evolve.Attachments AS at
        ON acq.order_id = at.order_id
    LEFT JOIN (
        SELECT * FROM transitions WHERE entity_type = 'ATTACHMENT'
    ) AS ast ON ast.entity_id = at.id
    WHERE at.id IS NOT NULL
)

--------------------------------------------------------------------------------------------------
-- Item and Attachment VAS
--------------------------------------------------------------------------------------------------
, vas_base_item_attachment AS (
    SELECT *
    FROM (
        SELECT
            vas.id, vas.entity_id, vas.entity_type, vas.user_action_type,
            vas.state, vas.name, vas.type,
            CAST(i.user_details_displayid AS STRING) AS fur_id,
            vas.start_date, vas.end_date,
            ROUND(DATEDIFF(vas.end_date, vas.start_date) / 30.45) AS tenures,
            vas.created_at,
            i.order_id
        FROM furlenco_silver.order_management_systems_evolve.value_added_services AS vas
        JOIN furlenco_silver.order_management_systems_evolve.items i ON i.id = vas.entity_id
        WHERE vas.entity_type = 'ITEM'
          AND vas.state NOT IN ('CANCELLED')
          AND vas.user_action_type = 'CART_CHECKOUT'

        UNION ALL

        SELECT
            vas.id, vas.entity_id, vas.entity_type, vas.user_action_type,
            vas.state, vas.name, vas.type,
            CAST(a.user_details_displayid AS STRING) AS fur_id,
            vas.start_date, vas.end_date,
            ROUND(DATEDIFF(vas.end_date, vas.start_date) / 30.45) AS tenures,
            vas.created_at,
            a.order_id
        FROM furlenco_silver.order_management_systems_evolve.value_added_services AS vas
        JOIN furlenco_silver.order_management_systems_evolve.attachments a ON a.id = vas.entity_id
        WHERE vas.entity_type = 'ATTACHMENT'
          AND vas.state NOT IN ('CANCELLED')
          AND vas.user_action_type = 'CART_CHECKOUT'
    ) AS finals
)

, vas_info AS (
    SELECT
        v.entity_id, v.entity_type,
        COALESCE(DATE(v.start_date), DATE(from_utc_timestamp(v.created_at, 'Asia/Kolkata'))) AS vas_date,
        SUM(COALESCE(CASE WHEN v.type = 'FURLENCO_CARE_PROGRAM'
                          THEN CAST(REPLACE(monetary_components_taxableamount, ',', '') AS DOUBLE)
                               / NULLIF(CAST(v.tenures AS DOUBLE), 0)
                     END, 0)) AS splitted_taxableAmount,
        SUM(COALESCE(CASE WHEN v.type = 'AC_INSTALLATION_CHARGE'
                          THEN CAST(REPLACE(monetary_components_taxableamount, ',', '') AS DOUBLE)
                               / NULLIF(CAST(v.tenures AS DOUBLE), 0)
                     END, 0)) AS Installation_charge,
        SUM(COALESCE(CASE WHEN v.type = 'DELIVERY_CHARGE'
                          THEN CAST(REPLACE(monetary_components_taxableamount, ',', '') AS DOUBLE)
                               / NULLIF(CAST(v.tenures AS DOUBLE), 0)
                     END, 0)) AS Delivery_charge
    FROM vas_base_item_attachment AS v
    JOIN furlenco_silver.furbooks_evolve.Revenue_Recognitions AS rr
        ON rr.accountable_entity_id = v.id AND rr.accountable_entity_type = 'VALUE_ADDED_SERVICE'
    WHERE rr.vertical = 'FURLENCO_RENTAL'
      AND rr.state IN ('PROCESSED', 'FUTURE')
      AND rr.deleted_at IS NULL
    GROUP BY ALL
)

, final_output_delivered_revenue AS (
    SELECT
        b.*,
        rr.start_date,
        CAST(REPLACE(monetary_components_taxableamount, ',', '') AS DOUBLE)                               AS Ebvmr_pre_tax,
        CAST(get_json_object(CAST(rr.monetary_components AS STRING), '$.tax.total') AS DOUBLE)            AS total_tax,
        (CAST(REPLACE(monetary_components_taxableamount, ',', '') AS DOUBLE)
         + CAST(get_json_object(CAST(rr.monetary_components AS STRING), '$.tax.total') AS DOUBLE))        AS Ebvmr_post_tax,
        (CAST(REPLACE(monetary_components_taxableamount, ',', '') AS DOUBLE)
         + COALESCE(vi.splitted_taxableAmount, 0)
         + COALESCE(vi.installation_charge, 0)
         + COALESCE(vi.delivery_charge, 0))                                                               AS ebvmr_with_vas,
        vi.splitted_taxableAmount                                                                          AS item_vas_monthly_pre,
        vi.Installation_charge,
        vi.Delivery_charge,
        vi.vas_date,
        get_json_object(CAST(rr.monetary_components AS STRING), '$.discounts[0].code')                    AS code_1,
        CAST(get_json_object(CAST(rr.monetary_components AS STRING), '$.discounts[0].amount') AS DOUBLE)  AS code_1_amount,
        get_json_object(CAST(rr.monetary_components AS STRING), '$.discounts[1].code')                    AS code_2,
        CAST(get_json_object(CAST(rr.monetary_components AS STRING), '$.discounts[1].amount') AS DOUBLE)  AS code_2_amount,
        CAST(get_json_object(CAST(rr.monetary_components AS STRING), '$.discounts[2].amount') AS DOUBLE)  AS code_3_amount,
        CAST(get_json_object(CAST(rr.monetary_components AS STRING), '$.discounts[3].amount') AS DOUBLE)  AS code_4_amount,
        CAST(get_json_object(CAST(rr.monetary_components AS STRING), '$.discounts[4].amount') AS DOUBLE)  AS code_5_amount,
        CAST(get_json_object(CAST(rr.monetary_components AS STRING), '$.discounts[5].amount') AS DOUBLE)  AS code_6_amount
    FROM base_item AS b
    JOIN furlenco_silver.furbooks_evolve.Revenue_Recognitions AS rr
        ON b.accountable_entity_id = rr.accountable_entity_id
        AND b.accountable_entity_type = rr.accountable_entity_type
    LEFT JOIN vas_info AS vi
        ON vi.entity_id = b.accountable_entity_id AND vi.entity_type = b.accountable_entity_type
    WHERE rr.state IN ('PROCESSED', 'FUTURE')
      AND rr.deleted_at IS NULL
    QUALIFY DENSE_RANK() OVER (PARTITION BY b.fur_id, rr.accountable_entity_id, rr.accountable_entity_type ORDER BY rr.start_date) = 1
)

SELECT
    bi.* EXCEPT (first_order_in_group, acquisition_date),
    ord.snapshotted_delivery_address_id,
    (COALESCE(bi.code_1_amount, 0) + COALESCE(bi.code_2_amount, 0) + COALESCE(bi.code_3_amount, 0)
     + COALESCE(bi.code_4_amount, 0) + COALESCE(bi.code_5_amount, 0) + COALESCE(bi.code_6_amount, 0)) AS total_discount_amount,
    CASE
        WHEN bi.code_2 IN ('FUR500') THEN 'FUR500'
        WHEN bi.code_1 IN ('FUR750') THEN 'FUR750'
        ELSE 'No'
    END                                                                                                  AS FUR_Coupon_flag,
    get_json_object(CAST(ord.payment_details AS STRING), '$.id')                                         AS payment_id,
    get_json_object(CAST(ord.payment_details AS STRING), '$.payable.total')                              AS entire_order_amt,
    get_json_object(CAST(ord.payment_details AS STRING), '$.payableAfterPaymentOffers.total')            AS entire_order_amt_post_ncemi,
    from_utc_timestamp(ord.created_at, 'Asia/Kolkata')                                                   AS order_created_date,
    sa.city,
    CASE WHEN bi.acquisition_date = bi.activation_date THEN 'New' ELSE 'Upsell' END                     AS flag_based_on_UA,
    pbp.unit_price                                                                                        AS asset_cost,
    CURRENT_TIMESTAMP + interval '330 minutes'                                                             AS refreshed_at
FROM final_output_delivered_revenue AS bi
LEFT JOIN furlenco_silver.order_management_systems_evolve.Orders AS ord
    ON ord.id = bi.order_id
LEFT JOIN furlenco_silver.order_management_systems_evolve.Snapshotted_Addresses AS sa
    ON sa.id = ord.snapshotted_delivery_address_id
LEFT JOIN product_base_price AS pbp
    ON pbp.catalog_product_id = bi.catalog_item_id;

### **Task 2: Validate Staging — Automated Quality Checks — the 4 validation checks**

In [0]:
%sql
-- ============================================================================
-- TASK 2: VALIDATE STAGING
-- ============================================================================
-- PURPOSE  : Runs 4 automated checks on the staging table before it is
--            promoted to live. If any check fails, the query raises an error
--            which causes the Databricks Workflow task to fail and blocks
--            Task 3 (the swap) from running entirely.
--
-- CHECKS   :
--   1. Empty check     — staging must have at least 1 row
--   2. Row count check — staging must have >= 90% of rows vs live table
--                        (guards against upstream CDC lag / source outages)
--   3. Schema check    — all columns in live must exist in staging
--                        (guards against upstream schema drift)
--   4. Null check      — key business columns must not be entirely NULL
--                        (guards against silent join failures)
--
-- DEPENDS ON : task1_write_staging.sql (set as dependency in Workflow)
-- NEXT TASK  : task3_swap_and_cleanup.sql
-- ============================================================================

-- FALLBACK: No-op if live table already exists, creates empty shell on first run
CREATE TABLE IF NOT EXISTS furlenco_analytics.materialized_tables.finance_fin_metrics
AS
SELECT *
FROM furlenco_analytics.materialized_tables.finance_fin_metrics_staging
WHERE 1 = 0;

-- ----------------------------------------------------------------------------
-- CHECK 1: Staging must not be empty
-- ----------------------------------------------------------------------------
SELECT
  CASE
    WHEN COUNT(*) = 0
    THEN RAISE_ERROR('VALIDATION FAILED: Staging table is empty. Swap blocked.')
  END
FROM furlenco_analytics.materialized_tables.finance_fin_metrics_staging;

-- ----------------------------------------------------------------------------
-- CHECK 2: Staging row count must be >= 90% of live table row count
-- Adjust 0.90 threshold if your data has known seasonal dips > 10%
-- ----------------------------------------------------------------------------
SELECT
  CASE
    WHEN staging_count < (live_count * 0.90)
    THEN RAISE_ERROR(
      CONCAT(
        'VALIDATION FAILED: Row count check failed. ',
        'Staging rows: ', CAST(staging_count AS STRING),
        ' | Live rows: ',  CAST(live_count    AS STRING),
        ' | Threshold (90%): ', CAST(CAST(live_count * 0.90 AS BIGINT) AS STRING),
        '. Swap blocked.'
      )
    )
  END
FROM (
  SELECT
    (SELECT COUNT(*) FROM furlenco_analytics.materialized_tables.finance_fin_metrics_staging) AS staging_count,
    (SELECT COUNT(*) FROM furlenco_analytics.materialized_tables.finance_fin_metrics)         AS live_count
);


-- ----------------------------------------------------------------------------
-- CHECK 3: Schema match — no columns should be missing from staging vs live
-- Catches upstream column drops or renames that would break downstream queries
-- ----------------------------------------------------------------------------
SELECT
  CASE
    WHEN COUNT(*) > 0
    THEN RAISE_ERROR(
      CONCAT(
        'VALIDATION FAILED: Schema mismatch detected. ',
        'Columns in live but missing from staging: ',
        ARRAY_JOIN(COLLECT_LIST(column_name), ', '),
        '. Swap blocked.'
      )
    )
  END
FROM (
  SELECT column_name
  FROM information_schema.columns
  WHERE table_catalog = 'furlenco_analytics'
    AND table_schema  = 'materialized_tables'
    AND table_name    = 'finance_fin_metrics'
  EXCEPT
  SELECT column_name
  FROM information_schema.columns
  WHERE table_catalog = 'furlenco_analytics'
    AND table_schema  = 'materialized_tables'
    AND table_name    = 'finance_fin_metrics_staging'
);


-- ----------------------------------------------------------------------------
-- CHECK 4: Key business columns must not be entirely NULL
-- Guards against silent upstream join failures producing all-NULL outputs
-- ----------------------------------------------------------------------------
SELECT
  CASE
    WHEN order_id_nulls      = total_rows THEN RAISE_ERROR('VALIDATION FAILED: order_id is entirely NULL in staging.')
    WHEN item_id_nulls       = total_rows THEN RAISE_ERROR('VALIDATION FAILED: item_id is entirely NULL in staging.')
    WHEN fur_id_nulls        = total_rows THEN RAISE_ERROR('VALIDATION FAILED: fur_id is entirely NULL in staging.')
    WHEN ebvmr_pre_tax_nulls = total_rows THEN RAISE_ERROR('VALIDATION FAILED: Ebvmr_pre_tax is entirely NULL in staging.')
  END
FROM (
  SELECT
    COUNT(*)                              AS total_rows,
    COUNT_IF(order_id      IS NULL)       AS order_id_nulls,
    COUNT_IF(accountable_entity_id       IS NULL)       AS item_id_nulls,
    COUNT_IF(fur_id        IS NULL)       AS fur_id_nulls,
    COUNT_IF(Ebvmr_pre_tax IS NULL)       AS ebvmr_pre_tax_nulls
  FROM furlenco_analytics.materialized_tables.finance_fin_metrics_staging
);


-- ----------------------------------------------------------------------------
-- All checks passed — log staging stats for the Workflow run audit trail
-- ----------------------------------------------------------------------------
SELECT
  'VALIDATION PASSED'                                                                                     AS status,
  (SELECT COUNT(*) FROM furlenco_analytics.materialized_tables.finance_fin_metrics_staging) AS staging_row_count,
  (SELECT COUNT(*) FROM furlenco_analytics.materialized_tables.finance_fin_metrics)         AS live_row_count,
  current_timestamp()   + interval '330 minutes'                                            AS validated_at;


### **Task 3: Atomic Swap + Cleanup — Promote to Live — atomic promotion, drop staging, optimize**

In [0]:
%sql
-- ============================================================================
-- TASK 3: ATOMIC SWAP + CLEANUP
-- ============================================================================
-- PURPOSE  : Promotes the validated staging table to live using CREATE OR
--            REPLACE — a single atomic Delta Lake operation that transitions
--            the live table from old snapshot to new with zero downtime.
--            Consumers querying during the operation always see the last
--            committed version (old data), never a "table not found" error.
--
-- STEPS    :
--   1. CREATE OR REPLACE live table from staging (atomic, zero gap)
--   2. Drop staging (no longer needed)
--   3. Stamp table properties with refresh metadata
--   4. OPTIMIZE + ZORDER for downstream query performance
--
-- NOTE     : CREATE OR REPLACE is a full data rewrite (slower than RENAME)
--            but is bulletproof — no millisecond gap where table is missing.
--            Delta ACID guarantees old snapshot is readable until commit.
--
-- DEPENDS ON : task2_validate_staging.sql (set as dependency in Workflow)
-- ============================================================================


-- ----------------------------------------------------------------------------
-- STEP 1: Atomic promotion — staging data replaces live in one Delta commit
-- During the write, consumers read the last committed version (old data).
-- At commit, Delta atomically flips to new snapshot. No gap, no error.
-- ----------------------------------------------------------------------------
CREATE OR REPLACE TABLE furlenco_analytics.materialized_tables.finance_fin_metrics
AS
SELECT *
FROM furlenco_analytics.materialized_tables.finance_fin_metrics_staging;


-- ----------------------------------------------------------------------------
-- STEP 2: Drop staging — it has been fully consumed into live
-- IF EXISTS is a safety net in case the job is re-run or staging was
-- already cleaned up by a previous partial run.
-- ----------------------------------------------------------------------------
DROP TABLE IF EXISTS
  furlenco_analytics.materialized_tables.finance_fin_metrics_staging;


-- ----------------------------------------------------------------------------
-- STEP 3: Stamp refresh metadata on the live table
-- EXECUTE IMMEDIATE is required because SET TBLPROPERTIES does not accept
-- function calls like current_timestamp() directly in Databricks.
-- Check freshness anytime with:
--   SHOW TBLPROPERTIES furlenco_analytics.materialized_tables.finance_fin_metrics;
-- ----------------------------------------------------------------------------
EXECUTE IMMEDIATE
  CONCAT(
    "ALTER TABLE furlenco_analytics.materialized_tables.finance_fin_metrics SET TBLPROPERTIES (",
    "'last_refreshed_at' = '", string(current_timestamp() + interval '330 minutes'), "', ",
    "'refresh_type' = 'full_refresh', ",
    "'refresh_schedule' = 'every_4_hours')"
  );


-- ----------------------------------------------------------------------------
-- STEP 4: Optimize the live table for downstream query performance
-- ZORDER co-locates data by the most commonly filtered/joined columns.
-- ----------------------------------------------------------------------------
OPTIMIZE furlenco_analytics.materialized_tables.finance_fin_metrics
  ZORDER BY (acquisition_type, current_entity_state, product_entity_type);


-- ----------------------------------------------------------------------------
-- Final confirmation — output appears in the Workflow run log
-- ----------------------------------------------------------------------------
SELECT
  'SWAP SUCCESSFUL'            AS status,
  COUNT(*)                     AS live_row_count,
  MIN(order_created_date)      AS earliest_order_created_date,
  MAX(order_created_date)      AS latest_order_created_date,
  current_timestamp() + interval '330 minutes' AS swapped_at
FROM furlenco_analytics.materialized_tables.finance_fin_metrics;

### **One-Time Setup: Create Stable View for Consumers — the one-time view creation**

In [0]:
%sql
-- ============================================================================
-- ONE-TIME SETUP
-- ============================================================================
-- PURPOSE  : Creates a stable view that all downstream consumers (dashboards,
--            reports, ad-hoc queries) must use instead of querying the raw
--            table directly.
--
-- WHY      : Task 3 drops and recreates the live table on every refresh run.
--            Any query hitting the raw table during that window gets a
--            "Table not found" error. A view always resolves to whatever
--            table the name currently points to, making the swap completely
--            invisible to consumers.
--
--
-- RUN      : Execute once manually before the first Workflow run.
--            Never put this inside the Workflow — it only needs to run once.
-- ============================================================================

-- CREATE OR REPLACE VIEW furlenco_gold.analytics.financial_metrics_finance_unified_analytics
-- AS 
-- SELECT * FROM furlenco_analytics.materialized_tables.finance_fin_metrics;

